In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


date_start = '2026-03-01'
date_end = '2026-04-30'

query_subscr = f'''
select
    contact_id,
    min(subscr_date_act::date) as first_subscribe_day
from subscr_status
group by contact_id
'''

subscr = su.execute_custom_query_gp(query_subscr)

subscr['first_subscribe_day'] = pd.to_datetime(
    subscr['first_subscribe_day']
)


scores = pd.concat(
    [
        march_scores.assign(score_month='2026-03'),
        april_scores.assign(score_month='2026-04')
    ],
    ignore_index=True
)

scores['score'] = pd.to_numeric(scores['score'], errors='coerce')

df = (
    scores
    .merge(subscr, on='contact_id', how='left')
)

df['is_subscribed'] = (
    df['first_subscribe_day'].between(
        pd.to_datetime(date_start),
        pd.to_datetime(date_end)
    )
).astype(int)


df['score_bin'] = pd.qcut(
    df['score'],
    q=10,
    duplicates='drop'
)

score_stats = (
    df
    .groupby(['score_month', 'score_bin'], as_index=False)
    .agg(
        clients_cnt=('contact_id', 'nunique'),
        subscribed_cnt=('is_subscribed', 'sum'),
        avg_score=('score', 'mean')
    )
)

score_stats['subscribed_pct'] = (
    score_stats['subscribed_cnt']
    / score_stats['clients_cnt']
    * 100
).round(2)

score_stats

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for month in score_stats['score_month'].unique():
    part = score_stats[score_stats['score_month'] == month]

    ax.plot(
        part['avg_score'],
        part['subscribed_pct'],
        marker='o',
        linewidth=2,
        label=month
    )

ax.set_title('Доля подписавшихся по бинам ML-score')
ax.set_xlabel('Средний score в бине')
ax.set_ylabel('Доля подписавшихся, %')
ax.grid(True, alpha=0.3)
ax.legend(title='Месяц score')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


date_start = '2026-03-01'
date_end = '2026-04-30'

query_subscr = f'''
select
    contact_id,
    min(subscr_date_act::date) as first_subscribe_day
from subscr_status
group by contact_id
'''

subscr = su.execute_custom_query_gp(query_subscr)

subscr['first_subscribe_day'] = pd.to_datetime(
    subscr['first_subscribe_day']
)


scores = pd.concat(
    [
        march_scores.assign(score_month='Март'),
        april_scores.assign(score_month='Апрель')
    ],
    ignore_index=True
)

scores['score'] = pd.to_numeric(
    scores['score'],
    errors='coerce'
)

df = scores.merge(
    subscr,
    on='contact_id',
    how='left'
)

df['is_subscribed'] = (
    df['first_subscribe_day'].between(
        pd.to_datetime(date_start),
        pd.to_datetime(date_end)
    )
).astype(int)


# бины score
df['score_bin'] = pd.qcut(
    df['score'],
    q=10,
    duplicates='drop'
)


score_stats = (
    df
    .groupby(['score_month', 'score_bin'], as_index=False)
    .agg(
        clients_cnt=('contact_id', 'nunique'),
        subscribed_cnt=('is_subscribed', 'sum')
    )
)

score_stats['subscribed_pct'] = (
    score_stats['subscribed_cnt']
    / score_stats['clients_cnt']
    * 100
).round(2)


# pivot для графика
plot_df = (
    score_stats
    .pivot(
        index='score_bin',
        columns='score_month',
        values='subscribed_pct'
    )
)

plot_df = plot_df.sort_index()

x = np.arange(len(plot_df))
width = 0.38

fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(
    x - width / 2,
    plot_df['Март'],
    width,
    label='Март'
)

ax.bar(
    x + width / 2,
    plot_df['Апрель'],
    width,
    label='Апрель'
)

ax.set_title(
    'Доля подписавшихся по бинам ML-score'
)

ax.set_xlabel('Бин score')
ax.set_ylabel('Доля подписавшихся, %')

ax.set_xticks(x)

ax.set_xticklabels(
    [str(v) for v in plot_df.index],
    rotation=45,
    ha='right'
)

ax.grid(True, axis='y', alpha=0.3)

ax.legend(title='Месяц score')

plt.tight_layout()
plt.show()